# 📓 Semana 7 · Dia 5 — LGPD/GDPR: direito ao esquecimento e retenção

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (parcial) + 🔑 TTL avançado |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (governança) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Política de privacidade documentada |

---


## 📖 Teoria — Privacidade no Lakehouse

Leis (LGPD/GDPR) exigem: consentimento, minimização, direito ao esquecimento, retenção limitada e auditoria. No Databricks:
- **Apagar**: `DELETE`, `DROP` (managed), ou reescrita com masking
- **TTL nativo** (DAIS 2026): expiração automática de dados por política
- **Retenção legal**: reter mínimo necessário (ex.: notas fiscais por 5 anos)
- **Anonimização**: substituir PII por pseudônimos na Prata/Ouro


## 📖 Teoria — Padrão: pseudonimização na Prata

O **Bronze** guarda o PII cru (append-only, auditoria). A **Prata** usa pseudônimos (IDs). O **Ouro** agrega sem PII. Quem precisa de PII acessa somente o Bronze com permissão e auditoria.


### 💻 Na prática — Pseudonimização

Crie uma versão da Prata sem PII (CustomerID → hash).


In [ ]:
# Pseudonimizar CustomerID (SHA-256) para a Prata
from pyspark.sql.functions import sha2, col
df = (spark.table("workspace.prata.fato_vendas")
    .withColumn("cliente_anon", sha2(col("CustomerID"), 256)))
df.select("cliente_anon").show(5, truncate=False)
print("Prata sem PII cru: apenas hash — reversível somente com a chave.")

### 💻 Na prática — Apagamento e retenção

Aplique TTL/delete para o direito ao esquecimento.


In [ ]:
%sql
-- Direito ao esquecimento: apagar registros de um cliente
DELETE FROM workspace.bronze.vendas_bronze WHERE CustomerID = '12345';
-- (em produção, logar a solicitação e reter a evidência)
SELECT COUNT(*) AS restantes FROM workspace.bronze.vendas_bronze;

In [ ]:
# TTL nativo (DAIS 2026) — expiração automática (recurso novo)
ttl_sql = """
-- Exemplo conceitual (conta paga):
ALTER TABLE workspace.bronze.vendas_bronze
  SET TBLPROPERTIES ('delta.dataTTL' = 'interval 5 years');
"""
print(ttl_sql)
print("TTL expira dados automaticamente após o período — compliance sem job manual.")

> 🎯 **Dica de prova**: Pergunta DEP/entrevista: 'como cumprir direito ao esquecimento com Delta?' → DELETE + VACUUM (apagar físico) ou reescrita com masking; TTL para retenção; pseudonimização na Prata. Na Free, DELETE e masking funcionam; TTL é pago.


## 🎯 Exercícios de fixação

**1.** Por que manter PII no Bronze e pseudônimo na Prata?

**2.** Qual a diferença entre DELETE (lógico) e VACUUM (físico)?

**3.** Desenhe sua política de retenção: o que reter, por quanto tempo, e como apagar.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** PII Bronze vs Prata

Bronze preserva a fonte (auditoria, reprocessamento); Prata distribui somente pseudônimo — minimização de PII em consumo.

**2.** DELETE vs VACUUM

DELETE marca a linha como removida (lógica, reversível via Time Travel); VACUUM remove o arquivo físico (irreversível).

**3.** Política

Ex.: Bronze retém 5 anos (auditoria fiscal) com TTL; Prata retém 2 anos; Ouro sem PII retém indefinido. Apagamento = DELETE + VACUUM + log da solicitação.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*